# Statistical Analysis

After response-based or connectivity-based hyperalignment has been completed, it is important to evaluate whether the alignment procedure has improved cross-subject functional correspondence. This tutorial introduces statistical utilities for summarizing, quantifying, and comparing hyperalignment results.

``fMRI-HA`` provides functions for computing **inter-subject correlation (ISC)** from response data, connectivity profiles, and representational geometry. It also includes tools for **between-subject multivariate pattern classification (bsMVPC)**, as well as basic utilities for **multivariate pattern analysis (MVPA)** and **representational similarity analysis (RSA)**. These functions are designed to support both routine quality assessment of hyperalignment outputs and downstream hypothesis-driven analyses.


## Script-Based Workflow

**Common script inputs**

| Item | Meaning |
|---|---|
| `work_dir` | Root HA work directory containing subject folders, `logs`, `masks`, and statistical outputs. |
| `sub_list` | Subject folder names, such as `['sub-01', 'sub-02']`. |
| `step` | Processing step containing the source files, usually `resample`, `combine`, or `align`. |
| `modality` | Source modality folder. For response-based statistics this is usually `response`. |
| `structure_name` | One or more structure folders, such as `hemi-L`, `hemi-R`, `volume-subcortical-L`, or `volume-mPFC`. |
| `file_filter` | BIDS-style filename filters used to select exactly one input file per subject and structure. Common keys include `run`, `ses`, `task`, `space`, `hemi`, `roi`, `desc`, and `suffix`. |
| `sls` | Searchlight definitions. Use a list for one structure or a dictionary keyed by hemisphere or volume structure. |


### (1) ISC

ISC measures cross-subject similarity. ``fMRI-HA`` provides response-based, connectivity-based, and representational-geometry ISC pipelines.

All three pipelines can compute leave-one-out ISC (`pairwise=False`) or pairwise ISC (`pairwise=True`). Results are saved as pickle files containing `raw` plus the selected summary field, usually `mean` or `median`.

If the data are already loaded as an array with shape `n_time x n_vertex/voxel x n_subject`, use `isc.isc_mat_nb` directly. This helper is useful for quick checks and for custom workflows that do not need the HA file layout.

Parameters of `isc.isc_mat_nb`:

| Parameter | Default | Meaning |
|---|---|---|
| `mat` | Required | 3D data matrix with shape `n_time x n_vertex/voxel x n_subject`. |
| `pairwise` | `False` | If `True`, computes ISC for every subject pair. If `False`, computes leave-one-out ISC. |
| `metric` | `'correlation'` | Similarity metric. Supported values are `'correlation'` and `'cosine'`. |
| `n_jobs` | `5` | Number of joblib workers. |
| `summary_statistic` | `'mean'` | Summary field to compute from `raw`; supported values are `'mean'` and `'median'`. |


In [ ]:
from pathlib import Path

import joblib
import neuroboros as nb
import nibabel as nib
import numpy as np

from fmriha import gensls
from fmriha.stat import isc

data_mat = joblib.load('path/to/matrix.pkl')

# mat shape: n_time x n_vertex/voxel x n_subject
results = isc.isc_mat_nb(
    mat=data_mat,
    pairwise=False,
    metric="correlation",
    n_jobs=5,
    summary_statistic="mean",
)

isc_map = results["ISC_mean"]


#### (a) Response ISC

`isc.response_isc_pipe` loads response matrices directly from the HA work directory and computes ISC for each selected structure. This is the most direct way to evaluate RHA outputs, for example by comparing raw response files under `resample/response` with aligned files under `align/response`.

Parameters of `isc.response_isc_pipe`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sub_list` | Required | Subject folder names included in ISC. |
| `step` | Required | Step folder used to locate input files, such as `resample` or `align`. |
| `modality` | Required | Modality folder used to locate input files, usually `response`. |
| `structure_name` | Required | Structure folder or list of folders to process. |
| `file_filter` | Required | BIDS-style filter used to select one `.npy` file per subject and structure. Files with a `split` tag are excluded automatically. |
| `pairwise` | `False` | If `True`, computes pairwise ISC. If `False`, computes leave-one-out ISC. |
| `metric` | `'correlation'` | ISC metric. The file-based pipeline currently supports `'correlation'`. |
| `njobs` | `5` | Number of workers used in the large-sample parallel path. |
| `chunk_size` | `2000` | Number of features processed per chunk. Reduce this when memory is limited. |
| `dtype` | `np.float32` | Numeric dtype for computed ISC arrays. |
| `summary_statistic` | `'mean'` | Summary statistic saved with `raw`; supported values are `'mean'` and `'median'`. |
| `json_path` | `None` | Defaults to `work_dir / "logs" / "prep_log.json"` when omitted. |
| `overwrite` | `False` | Whether matching JSON records are replaced. |
| `log_num` | `'00003'` | Identifier used in the progress-log filename. |
| `suffix` | `None` | Optional suffix appended to output filenames as `suffix-<suffix>`. |
| `time_split` | `None` | Optional segmentation. Use `None` for the full time series, `'continuous'` for first/second half, or `'interval'` for odd/even time points. |
| `plot_config` | `None` | Optional plotting control. `None` or `False` disables plotting; `True` saves default histograms; a dictionary can request histogram and surface figures. |


`plot_config` controls whether a statistical pipeline saves quick-look figures after the result pickle files are written. Use `None` or `False` to disable plotting. Use `True` to save default histogram figures. Use a dictionary to set the plotted result field, output folder, histogram style, surface style, display behavior, and plotting error behavior.

Histogram plotting can be used for any saved result file. Surface plotting is available for cortical results when the saved files include a matched `hemi-L` and `hemi-R` pair and both left and right surface files are provided. The default plotted field follows `summary_statistic`; set `method` to draw another field such as `median`.


In [ ]:
work_dir = Path("/path/to/ha_workdir")
sub_list = ["sub-rid000005", "sub-rid000014", "sub-rid000029"]
structures = ["hemi-L", "hemi-R"]

surface_mask = joblib.load("/path/to/onavg32_surface_mask.pkl")

plot_config = {
    "histogram": {
        "bins": 40,
        "color": "#4C72B0",
    },
    "surface": {
        "enabled": True,
        "lh": "/path/to/lh.surf.gii",
        "rh": "/path/to/rh.surf.gii",
        "surface_mask": surface_mask,  # optional dict keyed by "l" and "r"
        "color_range": (-0.2, 0.6),
        "cmap": "Reds",
    },
    "method": "mean",
    "plot_dir": "/path/to/output_figures",
    "show": False,
    "fail_on_error": False,
}

In [ ]:
# Example: response ISC for raw prepared response data.
isc.response_isc_pipe(
    work_dir=work_dir,
    sub_list=sub_list,
    step="resample",
    modality="response",
    structure_name=structures,
    file_filter={"run": "02"},
    njobs=5,
    suffix="raw",
    plot_config=plot_config,
)

# Example: response ISC for aligned response data.
isc.response_isc_pipe(
    work_dir=work_dir,
    sub_list=sub_list,
    step="align",
    modality="response",
    structure_name=structures,
    file_filter={"task": "rhaPRcortical", "level": "local"},
    njobs=5,
    suffix="rha",
    plot_config=plot_config,
)


#### (b) Connectivity ISC

Connectivity ISC first builds functional connectivity (FC) profiles with the same machinery used in CHA, then computes ISC across subjects on those FC profiles. This evaluates cross-subject similarity of subject-level connectivity profiles.

Parameters of `isc.fc_isc_pipe`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sub_list` | Required | Subject folder names included in ISC. |
| `step` | Required | Step folder containing response files used to build FC. |
| `modality` | Required | Source modality folder, usually `response`. |
| `seed_structure` | Required | One seed structure whose features define the FC output columns. |
| `target_structure` | Required | Target structure or list of target structures whose nodes are concatenated into FC profiles. |
| `file_filter` | Required | BIDS-style filter used to find seed response files. |
| `seed_sls` | Required | Searchlight definitions for seed nodes. |
| `target_sls` | Required | Searchlight definitions for target nodes. |
| `zscore` | `True` | If `True`, z-scores FC matrices before ISC. |
| `njobs` | `5` | Number of joblib workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `pairwise` | `False` | If `True`, computes pairwise ISC. If `False`, computes leave-one-out ISC. |
| `metric` | `'correlation'` | ISC metric. The file-based pipeline currently supports `'correlation'`. |
| `summary_statistic` | `'mean'` | Summary statistic saved with `raw`; supported values are `'mean'` and `'median'`. |
| `chunk_size` | `2000` | Number of FC features processed per chunk. |
| `dtype` | `np.float32` | Numeric dtype for FC and ISC computations. |
| `json_path` | `None` | Defaults to `work_dir / "logs" / "prep_log.json"` when omitted. |
| `overwrite` | `False` | Whether matching JSON records are replaced. |
| `log_num` | `'00003'` | Identifier used in the progress-log filename. |
| `suffix` | `None` | Optional suffix appended to output filenames and intermediate FC files. |
| `delete_fc` | `False` | If `True`, removes intermediate FC files after ISC is saved. |
| `time_split` | `None` | Optional segmentation. Use `None`, `'continuous'`, or `'interval'`. |
| `target_filter` | `None` | Optional BIDS-style filter for target response files. If `None`, `file_filter` is reused. |
| `plot_config` | `None` | Optional histogram and surface plotting configuration. |


**Searchlight preparation**

Connectivity ISC uses searchlight definitions for seed and target nodes. The representational-geometry ISC pipeline also uses searchlights to define the local response patterns from which representational geometry is computed.

For cortical surface data, first generate full-surface searchlights with `neuroboros`, then remove medial-wall vertices so that the searchlight indices match the prepared `.npy` arrays. The resulting `sls_surf` can be used directly by representational-geometry ISC. For connectivity-based ISC, reduce each searchlight to its center vertex to define vertex-wise `seed_sls` and `target_sls`.


In [ ]:
radius = 13
surface_space = "onavg-ico32"

# Generate searchlights on the full surface.
tmp = {
    lr: nb.sls(lr, radius, surface_space, mask=False, return_dists=True)
    for lr in "lr"
}
sls_lr = {lr: tmp[lr][0] for lr in tmp}
dists_lr = {lr: tmp[lr][1] for lr in tmp}

# The surface mask must correspond to the same surface space.
surface_mask = joblib.load("/path/to/onavg32_surface_mask.pkl")

sls_surf, dists_surf = gensls.sls_update(
    sls_lr,
    dists_lr,
    surface_mask,
    "lr",
)

# Connectivity statistics often use one seed/target node per vertex.
seed_sls = {
    lr: [arr[:1] for arr in arr_list]
    for lr, arr_list in sls_surf.items()
}
target_sls = {
    lr: [arr[:1] for arr in arr_list]
    for lr, arr_list in sls_surf.items()
}

del tmp, sls_lr, dists_lr


For CIFTI subcortical structures or NIfTI ROI structures, generate volume searchlights with `gensls.generate_searchlight_vol`. The dictionary keys should match the selected structure names, such as `volume-subcortical-L` or `volume-mPFC`. For connectivity statistics, use the center point of each volume searchlight as the seed/target definition in the same way as the surface example above.


In [ ]:
radius_vol = 4

# CIFTI subcortical example.
reference_cifti = "/path/to/reference.dtseries.nii"

mask_dense_subcortical = {
    lr: gensls.downsample_cifti_subcortical(
        cifti_file=reference_cifti,
        N=None,
        mask3d_out=None,
        sls_type="seed",
        voxel_sizes=None,
        drop_all_zero_columns=True,
        whole_mask=False,
        hemi_mask=True,
    )[lr]
    for lr in ["l", "r", "brain_stem"]
}

sls_info_subcortical = {
    lr: gensls.generate_searchlight_vol(
        mask_dense=mask_dense_subcortical[lr],
        mask_center=None,
        radius=radius_vol,
        threshold=0.2,
        njobs=10,
        verbose=5,
    )
    for lr in ["l", "r", "brain_stem"]
}

sls_subcortical = {
    f"volume-subcortical-{lr.upper() if lr in ['l', 'r'] else 'BRAIN_STEM'}": sls_info_subcortical[lr]["sls_idx"]
    for lr in ["l", "r", "brain_stem"]
}

# NIfTI ROI example.
roi_name = "mPFC"
nifti_mask_path = "/path/to/mPFC_mask.nii.gz"
mask_dense_roi = nib.load(nifti_mask_path).get_fdata().astype(bool)

sls_info_roi = gensls.generate_searchlight_vol(
    mask_dense=mask_dense_roi,
    mask_center=None,
    radius=radius_vol,
    threshold=0.2,
    njobs=10,
    verbose=5,
)

sls_roi = {f"volume-{roi_name}": sls_info_roi["sls_idx"]}

# Optional center-only definitions for connectivity statistics.
seed_sls_roi = {key: [arr[:1] for arr in value] for key, value in sls_roi.items()}
target_sls_roi = {key: [arr[:1] for arr in value] for key, value in sls_roi.items()}


In [ ]:
# seed_sls and target_sls should match the selected structures.
isc.fc_isc_pipe(
    work_dir=work_dir,
    sub_list=sub_list,
    step="resample",
    modality="response",
    seed_structure="hemi-L",
    target_structure=["hemi-L", "hemi-R"],
    file_filter={"run": "02"},
    target_filter={"run": "02"},
    seed_sls=seed_sls,
    target_sls=target_sls,
    njobs=5,
    suffix="raw",
    plot_config=plot_config,
)


#### (c) representational-geometry ISC

The representational-geometry ISC pipeline computes a local representational similarity matrix for each subject and searchlight, vectorizes that local geometry, and then evaluates cross-subject similarity of the geometry vectors. This analysis asks whether local representational structure is more consistent across subjects after alignment.

Parameters of `isc.rg_isc_pipe`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sub_list` | Required | Subject folder names included in ISC. |
| `step` | Required | Step folder containing response inputs. |
| `modality` | Required | Source modality folder, usually `response`. |
| `structure_name` | Required | Structure folder or list of folders to process. |
| `sls` | Required | Searchlight definitions used to build local representational geometry vectors. |
| `file_filter` | Required | BIDS-style filter used to select one response file per subject and structure. |
| `pairwise` | `False` | If `True`, computes pairwise RG-ISC. If `False`, computes leave-one-out RG-ISC. |
| `njobs` | `5` | Number of joblib workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `dtype` | `np.float32` | Numeric dtype used for saved values. |
| `summary_statistic` | `'mean'` | Summary statistic saved with `raw`; supported values are `'mean'` and `'median'`. |
| `json_path` | `None` | Defaults to `work_dir / "logs" / "prep_log.json"` when omitted. |
| `overwrite` | `False` | Whether matching JSON records are replaced. |
| `log_num` | `'00004'` | Identifier used in the progress-log filename. |
| `suffix` | `None` | Optional suffix appended to output filenames. |
| `time_split` | `None` | Optional segmentation. Use `None`, `'continuous'`, or `'interval'`. |
| `plot_config` | `None` | Optional histogram and surface plotting configuration. |


In [ ]:
isc.rg_isc_pipe(
    work_dir=work_dir,
    sub_list=sub_list,
    step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    sls=sls_surf,
    file_filter={"run": "02"},
    njobs=5,
    suffix="raw",
    plot_config=plot_config,
)


### (2) IDM

IDM reliability estimates whether the pattern of between-subject individual differences is stable across two independent temporal segments. For each searchlight, `fMRI-HA` constructs a subject-by-subject similarity matrix for each segment, vectorizes the upper triangle, and correlates the two vectors.

The same `idm.idm_pipe` function supports response, connectivity, and representational-geometry IDM reliability through the `method` argument.

```{image} pic/statistics/idm_workflow.png
:width: 800px
```

Parameters of `idm.idm_pipe`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sub_list` | Required | Subject folder names included in IDM reliability. |
| `step` | Required | Source step folder. For `method='response'`, this is also the IDM input location. |
| `modality` | Required | Source modality folder, usually `response`. |
| `structure_name` | Required | Structure folder or list of folders to process. For connectivity IDM, this should identify one seed structure. |
| `file_filter` | Required | BIDS-style filter used to locate source files. If `ses` is present, it is also used when locating split files. |
| `time_split` | Required | Split strategy. Use `'continuous'` for first/second half or `'interval'` for odd/even time points. |
| `sls` | Required | Searchlight definitions used to build IDM vectors from final split files. |
| `njobs` | `5` | Number of joblib workers. |
| `verbose` | `1` | Joblib verbosity level. |
| `method` | `'response'` | IDM representation. Supported values are `'response'`, `'connectivity'`, and `'representational_geometry'`. |
| `prepare_inputs` | `True` | If `True`, creates split-half input files before reliability estimation. If `False`, expects precomputed split files in the standard layout. |
| `target_structure` | `None` | Connectivity-specific target structure or structures used when `method='connectivity'` and `prepare_inputs=True`. |
| `seed_sls` | `None` | Connectivity-specific seed searchlights. If `None`, `sls` is reused. |
| `target_sls` | `None` | Connectivity-specific target searchlights. If `None`, `sls` is reused. |
| `zscore` | `True` | Connectivity-specific control for FC standardization during split preparation. |
| `target_filter` | `None` | Connectivity-specific BIDS-style target-file filter. If `None`, `file_filter` is reused. |
| `suffix` | `None` | Optional suffix appended to final IDM reliability filenames. |
| `dtype` | `np.float32` | Numeric dtype for prepared inputs and saved reliability values. |
| `summary_statistic` | `'mean'` | Summary statistic computed across the raw reliability vector; supported values are `'mean'` and `'median'`. |
| `delete_split_files` | `False` | If `True`, removes split files generated by this IDM run after final results are saved. |
| `json_path` | `None` | Defaults to `work_dir / "logs" / "prep_log.json"` when omitted. |
| `overwrite` | `False` | Whether matching JSON records are replaced. |
| `log_num` | `'00004'` | Identifier used in the progress-log filename. |

In [ ]:
import numpy as np
from fmriha.stat import idm

#### (a) Response IDM

Response IDM evaluates whether between-subject similarity patterns in local response activity are reliable across split time segments.


In [ ]:
# Response IDM reliability.
idm.idm_pipe(
    work_dir=work_dir,
    sub_list=sub_list,
    step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    file_filter={"run": "02"},
    time_split="continuous",
    sls=sls_surf,
    njobs=5,
    verbose=1,
    method="response",
    prepare_inputs=True,
    suffix="raw",
)

#### (b) Connectivity IDM

Connectivity IDM evaluates the split-segment reliability of individual-difference patterns derived from functional connectivity profiles.


In [ ]:
# Connectivity IDM reliability.
idm.idm_pipe(
    work_dir=work_dir,
    sub_list=sub_list,
    step="resample",
    modality="response",
    structure_name="hemi-L",
    file_filter={"run": "02"},
    time_split="continuous",
    sls=seed_sls,
    njobs=5,
    method="connectivity",
    prepare_inputs=True,
    target_structure=["hemi-L", "hemi-R"],
    seed_sls=seed_sls,
    target_sls=target_sls,
    target_filter={"run": "02"},
    suffix="raw",
)

#### (c) representational-geometry IDM

The representational-geometry IDM pipeline evaluates whether individual-difference patterns in local representational structure are stable across split time segments.


In [ ]:
# representational-geometry IDM reliability.
idm.idm_pipe(
    work_dir=work_dir,
    sub_list=sub_list,
    step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    file_filter={"run": "02"},
    time_split="continuous",
    sls=sls_surf,
    njobs=5,
    method="representational_geometry",
    prepare_inputs=True,
    suffix="raw",
)


### bsMVPC

bsMVPC evaluates temporal correspondence across subjects. For each subject, a sliding window from that subject is matched against sliding windows from the leave-one-out group mean. Accuracy is computed as the proportion of windows for which the matching window is correctly identified.

`bsmvpc.bsmvpc_pipeline` computes searchlight-wise bsMVPC from response data in the HA work directory layout.

Parameters of `bsmvpc.bsmvpc_pipeline`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sub_list` | Required | Subject folder names included in bsMVPC. |
| `step` | Required | Step folder containing response inputs, such as `resample` or `align`. |
| `modality` | Required | Source modality folder, usually `response`. |
| `structure_name` | Required | Structure folder or list of folders to process. |
| `file_filter` | Required | BIDS-style filter used to select one `.npy` file per subject and structure. |
| `window_length` | Required | Number of time points in each classification window. |
| `window_step` | Required | Step size between adjacent windows. |
| `sls` | Required | Searchlight definitions used to extract local response features. |
| `njobs` | `5` | Number of joblib workers across searchlights. |
| `verbose` | `1` | Joblib verbosity level. |
| `dtype` | `np.float32` | Numeric dtype for computed values. |
| `summary_statistic` | `'mean'` | Summary statistic saved with `raw`; supported values are `'mean'` and `'median'`. |
| `json_path` | `None` | Defaults to `work_dir / "logs" / "prep_log.json"` when omitted. |
| `overwrite` | `False` | Whether matching JSON records are replaced. |
| `log_num` | `'00003'` | Identifier used in the progress-log filename. |
| `suffix` | `None` | Optional suffix appended to output filenames. |
| `plot_config` | `None` | Optional histogram and surface plotting configuration. |


In [ ]:
import numpy as np
from fmriha.stat import bsmvpc

bsmvpc.bsmvpc_pipeline(
    work_dir=work_dir,
    sub_list=sub_list,
    step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    file_filter={"run": "02"},
    window_length=6,
    window_step=1,
    sls=sls_surf,
    njobs=5,
    verbose=1,
    suffix="raw",
)


### MVPA (with Task Data)

In addition to ISC, bsMVPC, and IDM-based reliability analyses, ``fMRI-HA`` also provides basic utilities for task-based multivariate pattern analysis (MVPA). These tools can be used to evaluate whether distributed neural patterns contain information about experimental conditions, stimulus categories, or other task labels.

Unlike the pipeline-based statistical utilities introduced above, the MVPA tools do not require data to be organized strictly according to the HA work directory layout. Users only need to prepare the feature matrix, labels, and grouping variables in the required format before running the analysis. The underlying classification procedures are implemented using `scikit-learn`, a mature and widely used machine-learning library.

In this section, we introduce the MVPA module in ``fMRI-HA``. The current implementation focuses on support vector machine (SVM) classification, a widely used and well-established approach in neuroimaging MVPA.

The module provides a nested cross-validation framework. Users may optionally perform hyperparameter tuning in the inner loop, while the outer loop is used to estimate generalization performance. Different cross-validation schemes are available depending on the structure of the dataset, such as subject-wise, run-wise, or combined subject-run-wise splitting. Below, we first introduce the supported cross-validation schemes and then demonstrate the basic usage.

**Supported Cross-Validation Schemes**

``fMRI-HA`` currently supports three cross-validation schemes: **subject-level**, **run-level**, and **subject-run-level**. These schemes differ in how the training and test sets are separated.

**Subject-Level CV**

In subject-level cross-validation, entire subjects are held out for testing. All samples from the held-out subjects form the test set, while samples from the remaining subjects are used for training.

**Use case:** This scheme is appropriate when the goal is to evaluate generalization across individuals, especially when each subject has only a limited number of runs.

**Run-Level CV**

In run-level cross-validation, entire runs are held out for testing across all subjects. Samples from the selected runs form the test set, while samples from the remaining runs form the training set.

**Use case:** This scheme should be used with caution. Because the same subjects may contribute data to both training and test sets, subject-specific information may be shared across folds. It is mainly suitable when run-wise generalization is the main research target.

**Subject-Run-Level CV**

In subject-run-level cross-validation, training and test sets are separated jointly by subject and run. For example, the training set may contain one run from a subset of subjects, whereas the test set contains another run from different subjects.

**Use case:** This scheme provides a stricter evaluation of generalization by reducing the chance that subject-specific or run-specific information is shared between training and test sets. It is recommended when the dataset contains enough subjects and runs to support this type of split.

In practice, **subject-run-level CV** is recommended when sufficient runs are available. If the dataset contains only a small number of runs per subject, **subject-level CV** is often a reasonable alternative. **Run-level CV** should generally be reserved for specific experimental designs where run-wise generalization is the main target.

Schematics of the three cross-validation schemes are shown below.


```{image} pic/statistics/mvpa1.png
:width: 800px
```

```{image} pic/statistics/mvpa2.png
:width: 800px
```

```{image} pic/statistics/mvpa3.png
:width: 800px
```

The example below uses task beta maps from the Forrest dataset. Each row in `input_df` corresponds to one beta map: one subject, one run, and one condition. The example reuses `sls_surf` and `surface_mask` from the searchlight preparation section above.

In [ ]:
import os
from pathlib import Path

import neuroboros as nb
import numpy as np
import pandas as pd

from fmriha.preprocessing import fileio as fio
from fmriha.stat.mvpa import cv_nested

os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

work_dir = Path("/path/to/tutorial_4")
os.chdir(work_dir)

sub_list = sorted([idx for idx in os.listdir(work_dir) if idx.startswith("sub")])
lr = "l"
sls_mvpa = sls_surf[lr]
cortical_mask_lr = {
    hemi: nb.mask(hemi, "onavg-ico32")
    for hemi in "lr"
}

We use a pandas DataFrame to organize the input data, as illustrated in the figure below. Each row corresponds to a single observation, for example, a beta map from a specific condition and run for a given subject. The feature associated with each row should be either a vector of length `n_v` or a matrix of shape `1 x n_v`, where `n_v` denotes the number of vertices or voxels.

The DataFrame follows a fixed column schema:
- **feature**: the neural data used for MVPA (vector or 1 x n_v matrix)  
- **label**: the condition or class label  
- **subject_id**: subject identifier  
- **run_id**: run identifier

```{image} pic/statistics/mvpa4.png
:width: 800px
```

Although the column names are fixed, the input format is flexible with respect to spatial resolution. In our demo, each feature is a `1 x 9675` vector representing a whole-hemisphere beta map. MVPA is then performed using a searchlight approach. For a given searchlight, for example, `sl = [1, 119, 25, 35, ...]`, the function extracts the corresponding vertices from the feature vector and uses them as the input for classification within that local unit.

This design also naturally extends to ROI-based analyses. For example, if only a single ROI is used, the feature may be a `1 x 119` vector, and the searchlight list can simply be defined as `[[0, 1, 2, ..., 118]]`. In general, the searchlight definition (`sls`) is a list of lists, where each inner list specifies the indices of vertices or voxels included in one MVPA unit, whether it represents a searchlight or an ROI.

In [ ]:
rows = []
for sub in sub_list:
    for run in ["1", "2", "3", "4"]:
        for cope in ["1", "2", "3", "4", "5", "6"]:
            source_file = (
                work_dir
                / sub
                / f"GLM/run-{run}/beta_map/{lr}_SurfaceStats/cope{cope}.func.gii"
            )
            source_data = fio.load_gifti_data(source_file)
            feature = source_data[:, cortical_mask_lr[lr]]
            rows.append(
                {
                    "feature": feature,
                    "label": f"condition{cope}",
                    "subject_id": sub,
                    "run_id": f"run-{run}",
                }
            )

input_df = pd.DataFrame(rows)

print(f"Prepared {len(input_df)} beta maps.")
print(f"Subjects: {input_df['subject_id'].nunique()}")
print(f"Runs: {input_df['run_id'].nunique()}")
print(f"Conditions: {input_df['label'].nunique()}")
print(f"Feature shape per sample: {input_df.loc[0, 'feature'].shape}")


**Cross-validation configuration**

The outer CV loop estimates model performance. The inner CV loop selects the SVM regularization parameter. In this example, both subject and run are left out in the outer loop, which tests generalization across held-out subjects and held-out runs. If `inner_cv["level"]` is `False`, the first value in `param_grid_C` is used.


In [ ]:
outer_cv = {
    "level": True,
    "sub_level": True,
    "sub_cv_frame": "loo",
    "sub_cv_split": None,
    "run_level": True,
    "run_cv_frame": "loo",
    "run_cv_split": None,
    "kernel": "linear",
}

inner_cv = {
    "level": True,
    "sub_level": True,
    "sub_cv_frame": "loo",
    "sub_cv_split": None,
    "run_level": True,
    "run_cv_frame": "loo",
    "run_cv_split": None,
    "kernel": "linear",
}


Parameters of `cv_nested.leave_sub_run_out`:

| Parameter | Default | Meaning |
|---|---|---|
| `input_df` | Required | DataFrame containing `feature`, `label`, `subject_id`, and `run_id`. |
| `outer_cv` | Required | Outer subject/run CV configuration dictionary used for final model evaluation. |
| `inner_cv` | Required | Inner subject/run CV configuration dictionary used for hyperparameter selection. |
| `sls` | Required | Searchlight or ROI index lists. Each list selects features for one local MVPA unit. |
| `param_grid_C` | `(1, 10)` | Candidate SVM `C` values searched in the inner loop. |
| `evaluate` | `'accuracy'` | Metric used for model selection and output summaries. Supported values include `'accuracy'` and `'auc'`. |
| `random_seed` | `42` | Base random seed used by cross-validation splitters. |
| `n_jobs` | `25` | Number of parallel jobs across searchlights. |
| `verbose` | `0` | Joblib verbosity level. |

Parameters of `cv_nested.leave_sub_run_out_permute`:

| Parameter | Default | Meaning |
|---|---|---|
| `input_df` | Required | DataFrame containing `feature`, `label`, `subject_id`, and `run_id`. |
| `outer_cv` | Required | Outer subject/run CV configuration dictionary used for observed and permuted evaluation. |
| `inner_cv` | Required | Inner subject/run CV configuration dictionary. During permutations, the implementation disables inner-loop tuning for the permuted runs. |
| `sls` | Required | Searchlight or ROI index lists. |
| `param_grid_C` | `(1, 10)` | Candidate SVM `C` values. |
| `evaluate` | `'accuracy'` | Metric used for permutation significance testing. Supported values include `'accuracy'` and `'auc'`. |
| `random_seed` | `42` | Base random seed used for observed CV and label permutations. |
| `n_jobs` | `25` | Number of parallel jobs across searchlights. |
| `n_perm` | `1000` | Number of label permutations. Use a larger value for final inference. |
| `permute_labels` | `'subject'` | Label permutation level. Supported values are `'global'`, `'subject'`, and `'subject_run'`. |


**Nested CV MVPA without permutation**

The following call runs nested CV for the selected hemisphere. For a quick demo, only the first 100 searchlights are used. For a full analysis, pass the complete `sls_mvpa` list and choose a broader `param_grid_C` when needed.


In [ ]:
results_output_hemi = cv_nested.leave_sub_run_out(
    input_df=input_df,
    outer_cv=outer_cv,
    inner_cv=inner_cv,
    sls=sls_mvpa[:100],
    param_grid_C=[1, 10],
    evaluate="accuracy",
    random_seed=42,
    n_jobs=20,
    verbose=1,
)


**Permutation test**

Permutation testing estimates an empirical null distribution by shuffling labels. The example uses `n_perm=20` for speed. For final inference, use a substantially larger number, such as `n_perm >= 1000`.


In [ ]:
ob, null_val, p = cv_nested.leave_sub_run_out_permute(
    input_df=input_df,
    outer_cv=outer_cv,
    inner_cv=inner_cv,
    sls=sls_mvpa[:100],
    param_grid_C=[1, 10],
    evaluate="accuracy",
    random_seed=42,
    n_jobs=20,
    n_perm=20,
    permute_labels="subject",
)


**Quick summary**

The output contains searchlight-wise metrics. The example below prints a compact summary of observed accuracy and permutation results.


In [ ]:
acc = results_output_hemi["accuracy_mean"]

print("\nMVPA summary")
print("------------")
print(f"Number of searchlights: {acc.size}")
print(f"Mean accuracy: {np.nanmean(acc):.4f}")
print(f"Median accuracy: {np.nanmedian(acc):.4f}")
print(f"Max accuracy: {np.nanmax(acc):.4f}")

print("\nPermutation summary")
print("-------------------")
print(f"Observed mean accuracy: {np.nanmean(ob['accuracy_mean']):.4f}")
print(f"Mean null accuracy: {np.nanmean(null_val):.4f}")
print(f"Significant searchlights, p < 0.05: {np.sum(p < 0.05)}")
print(f"Minimum p-value: {np.nanmin(p):.4f}")


### Background FC from Task Data

Many studies only include task-fMRI data, and the order of stimulus presentation may vary across participants. In such cases, directly computing FC from task time series may introduce task-related confounds or temporal mismatches across subjects.

A common solution is to estimate **background functional connectivity (background FC)** by first regressing out task-related effects with a GLM. In `fMRI-HA`, `glm_pipe` fits a subject-level GLM to preprocessed time-by-feature response matrices and can save the residual time series under `sub-*/glm/residuals/<structure>`. These residuals are then used to construct FC profiles for CHA.

The example below follows the task-GLM workflow shown in the connectivity HA tutorial. `events_tsv` can be a list, with one events file per subject in the same order as `sub_list`, or a single events file if all subjects share the same design.

Parameters of `glm_pipe`:

| Parameter | Default | Meaning |
|---|---|---|
| `work_dir` | Required | Root HA work directory. |
| `sub_list` | Required | Subject folder names to process. |
| `events_tsv` | Required | Events TSV file or list of files with `onset`, `duration`, and `trial_type` columns. A single file is shared by all subjects; a list is matched to `sub_list` by order. |
| `input_pattern` | `None` | Optional relative input path template containing `{sub}`. If provided, it is used directly instead of `step`, `modality`, `structure_name`, and `file_filter`. |
| `output_types` | `('beta', 'residuals')` | Outputs to save. Supported values are `'beta'` and `'residuals'`. |
| `condition_names` | `None` | Conditions to save as beta vectors. If `None`, conditions are inferred from the `trial_type` column. |
| `t_r` | Required | Repetition time used to build the task design matrix. |
| `hrf_model` | `'spm + derivative'` | Hemodynamic response model passed to the design-matrix builder. |
| `drift_model` | `'cosine'` | Drift model passed to the design-matrix builder. Use `None` to omit drift regressors. |
| `high_pass` | `0.01` | High-pass cutoff used with compatible drift models. Use `None` to omit high-pass filtering. |
| `fir_delays` | `None` | FIR delays used when an FIR HRF model is selected. |
| `step` | `None` | Pipeline step folder used to locate input files when `input_pattern=None`, such as `resample`. |
| `modality` | `None` | Modality folder used to locate input files when `input_pattern=None`, usually `response`. |
| `structure_name` | `None` | Structure folder or list of folders to process when `input_pattern=None`. |
| `file_filter` | `None` | BIDS-style filter used to select one input `.npy` file per subject and structure when `input_pattern=None`. |
| `confounds_pattern` | `None` | Optional relative path template containing `{sub}` for confound files. Supported formats are `.npy`, `.tsv`, and `.csv`. |
| `add_reg_names` | `None` | Confound column names. For `.tsv` or `.csv`, these columns are selected; for `.npy`, they name the columns. |
| `noise_model` | `'ols'` | GLM noise model. Supported values include `'ols'` and `'ar1'`. |
| `standardize_data` | `False` | If `True`, z-scores each feature across time before fitting. |
| `include_derivatives` | `False` | If `True`, saves derivative or FIR-delay beta vectors with explicit names. |
| `suffix` | `None` | Optional suffix appended to GLM output filenames. |
| `n_jobs` | `1` | Number of subject-level joblib workers. |
| `verbose` | `1` | Verbosity level for progress messages and joblib. |
| `overwrite` | `False` | If `False`, a subject/structure is skipped when planned output files already exist. |
| `log_num` | `'00004'` | Identifier used in the progress-log filename. |

In [ ]:
from pathlib import Path
import numpy as np
from fmriha import ha
from fmriha.stat import glm_pipe
import neuroboros as nb


work_dir = Path("/path/to/task_ha_workdir")
sub_list = ["sub-01", "sub-02", "sub-03"]

events_tsv = [
    "/path/to/sub-01_task-objectcategories_run-1_events.tsv",
    "/path/to/sub-02_task-objectcategories_run-1_events.tsv",
    "/path/to/sub-03_task-objectcategories_run-1_events.tsv",
]

glm_params = {
    "output_types": ("beta", "residuals"),     # Save condition betas and residual time series.
    "condition_names": None,                   # Infer conditions from the trial_type column.
    "t_r": 2.5,                                # Repetition time of the task-fMRI data.
    "hrf_model": "spm + derivative",
    "drift_model": None,
    "high_pass": None,
    "noise_model": "ols",
    "standardize_data": False,                 # Set to True only if the input data need z-scoring here.
    "include_derivatives": False,              # Do not save derivative beta maps as separate outputs.
    "suffix": "object"                        # Suffix appended to GLM output filenames.
}

glm_summary = glm_pipe(
    work_dir=work_dir,
    sub_list=sub_list,
    events_tsv=events_tsv,
    step="resample",
    modality="response",
    structure_name=["hemi-L", "hemi-R"],
    file_filter={"run": "01"},
    n_jobs=2,
    verbose=1,
    overwrite=False,
    **glm_params,
)

After the residuals have been saved, use `ha.fc_build` to construct background-FC profiles. Here, both seed and target data are read from `step="glm"` and `modality="residuals"`. The saved FC files can then be used as connectivity inputs for Template, T Matrix, and Alignment calculations in the CHA workflow; those later HA calls are omitted here.

In [ ]:
surface_sls_lr = {
    lr: nb.sls(
        lr,
        radius=20,
        space="onavg-ico32",
        mask=True,
        return_dists=True,
    )
    for lr in "lr"
}

sls_surf = {lr: surface_sls_lr[lr][0] for lr in surface_sls_lr}
dists_surf = {lr: surface_sls_lr[lr][1] for lr in surface_sls_lr}

sls_surf_seed = {
    lr: [np.array([idx[0]]) for idx in sls_surf[lr]]
    for lr in "lr"
}

sls_surf_target = {
    lr: nb.sls(
        lr,
        radius=13,
        space="onavg-ico32",
        center_space="onavg-ico8",
        mask=True,
        return_dists=False,
    )
    for lr in "lr"
}

glm_residual_filter = {"run": "01", "desc": "glm", "suffix": "object"}

for seed_structure in ["hemi-L", "hemi-R"]:
    ha.fc_build(
        work_dir=work_dir,
        sub_list=sub_list,
        seed_step="glm",
        seed_modality="residuals",
        seed_structure=seed_structure,
        seed_file_filter=glm_residual_filter,
        seed_sls=sls_surf_seed,
        target_step="glm",
        target_modality="residuals",
        target_structure=["hemi-L", "hemi-R"],
        target_file_filter=glm_residual_filter,
        target_sls=sls_surf_target,
        zscore=True,
        njobs=5,
        verbose=1,
        dtype=np.float32,
        overwrite=False,
        log_num="task",
        suffix="bgfc",
    )

## GUI Workflow

Launch the graphical tools from Python. All statistical analyses will be performed using the **Utilities** subpage.

In [ ]:
from fmriha import gui
gui()

```{image} pic/data_prep/gui.png
:width: 800px
```

### (1) ISC

#### (a) Response ISC

First, click the **ISC of Response** button to enter the Response ISC subpage.

After entering the Work Dir, the program will automatically search for the Sub List under that directory, which can be modified as needed.

**Surface Plot** is optional. When enabled, the corresponding histogram or brain surface map will be generated after the computation is completed. If you would like to generate a brain surface map, you must provide the corresponding hemisphere's `.surf.gii` file and a **Cortical Mask** with the medial wall removed. Please note that the provided **Cortical Mask** must satisfy the following condition: retained surface positions must be marked as `True`, and positions to be masked out, such as the medial wall region, must be marked as `False`. Currently, only `.pkl` format files are supported. The required content structure is:

In [ ]:
{
    'l': left_mask,
    'r': right_mask
}

```{image} pic/statistics/gui_responseISC.png
:width: 800px
```

#### (b) Connectivity ISC

Click the **ISC of Connectivity** button to enter the Connectivity ISC subpage. The related operations are similar to those in **ISC of Response**, except that the searchlights need to be reconstructed.

For the meanings of Seed and Target, please refer to the **Connectivity Calculation** page. The construction process for both Surface and Volume searchlights is exactly the same as the operations described in **Space and Searchlight Configuration** on the main page.

The figure below uses cortical data from hemi-L and hemi-R as examples:

```{image} pic/statistics/gui_fcISC_1.png
:width: 800px
```

```{image} pic/statistics/gui_fcISC_2.png
:width: 800px
```

#### (c) representational-geometry ISC

The calculation procedure for representational-geometry ISC is similar. Simply click **ISC of RDM** and follow the same operations. The figure below again uses cortical data from hemi-L and hemi-R as examples.

```{image} pic/statistics/gui_rdmISC.png
:width: 800px
```

### (2) IDM

The calculation procedures for the three types of IDM are very similar to those for ISC. Simply click the corresponding button and fill in the required information as shown below. The following figures again use cortical data from hemi-L and hemi-R as examples.


#### (a) Response IDM

```{image} pic/statistics/gui_responseIDM.png
:width: 800px
```

#### (b) Connectivity IDM

```{image} pic/statistics/gui_fcIDM_1.png
:width: 800px
```

```{image} pic/statistics/gui_fcIDM_2.png
:width: 800px
```

#### (c) representational-geometry IDM

```{image} pic/statistics/gui_rdmIDM.png
:width: 800px
```

### (3) bsMVPC

Click the **bsMVPC** button to enter the subpage, then fill in the required information as shown below. Among the parameters, only **Window Length** and **Window Step** are unique to this module and can be adjusted as needed.

The figure below again uses cortical data from hemi-L and hemi-R as examples.

```{image} pic/statistics/gui_bsmvpc.png
:width: 800px
```